# 01 · The KV block pool: paging, ref counts and LRU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/vllm/01_block_pool.ipynb) &nbsp; 📖 Concept deck: [KV cache manager and PagedAttention](https://nilesh-salpe.github.io/visual-ai/vllm/03-kv-cache-manager/) &nbsp; ⏱️ Suggested time: **35 min**

**You will build:** a pool of fixed-size KV blocks with a free list, per-request block tables, reference counts for shared blocks, and LRU reuse of freed blocks, the design of `vllm/v1/core/block_pool.py`.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


from collections import OrderedDict
import math
BLOCK = 16          # vLLM's CacheConfig.DEFAULT_BLOCK_SIZE

## Task 1: How many blocks?

A request with `n` tokens needs ⌈n / block_size⌉ blocks; the last one may be partly empty. Return `(blocks, wasted_slots)` where wasted slots are the unused token slots in the last block.

In [ ]:
def blocks_needed(n, block=BLOCK):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_blocks_needed(f):
    assert f(1) == (1, 15) and f(16) == (1, 0) and f(17) == (2, 15) and f(1000) == (63, 8)

check("blocks_needed", _test_blocks_needed, blocks_needed)

## Task 2: A block pool with ref counts and an LRU free queue

Implement `BlockPool(num_blocks)` with:
* `allocate(k)` → list of k block ids taken from the **front** of the free queue (the least recently freed), or `None` if fewer than k are free. Allocated blocks get ref count 1.
* `share(block_ids)` → increments ref counts (another request reuses these blocks, e.g. a cached prefix).
* `free(block_ids)` → decrements ref counts; blocks reaching 0 go to the **back** of the free queue. vLLM frees a request's blocks in **reverse** order, so the tail of a sequence is evicted before its head.
* `num_free()`.

In [ ]:
class BlockPool:
    def __init__(self, num_blocks):
        self.free_q = OrderedDict((i, None) for i in range(num_blocks))
        self.ref = [0] * num_blocks
    def allocate(self, k):
        # TODO
        raise NotImplementedError
    def share(self, ids):
        # TODO
        raise NotImplementedError
    def free(self, ids):
        # TODO (iterate in reverse)
        raise NotImplementedError
    def num_free(self):
        return len(self.free_q)

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_BlockPool(cls):
    p = cls(8)
    a = p.allocate(3); b = p.allocate(2)
    assert a == [0, 1, 2] and b == [3, 4] and p.num_free() == 3
    assert p.allocate(4) is None, "not enough free blocks"
    p.share(a[:2])                      # another request reuses the first two blocks
    p.free(a)
    assert p.num_free() == 4, "only block 2 reached ref count 0"
    p.free(a[:2])
    assert p.num_free() == 6
    assert list(p.free_q)[-3:] == [2, 1, 0], "freed in reverse order: tail first in eviction order"

check("BlockPool", _test_BlockPool, BlockPool)

## Task 3: Paged vs contiguous reservation

Before paging, servers reserved a contiguous region of `max_len` token slots per request. Given the actual final lengths of a set of requests, return `(contiguous_waste, paged_waste)`: the fraction of reserved slots never used, for (a) reserving `max_len` each and (b) paging with 16-token blocks.

In [ ]:
def paged_vs_contiguous(lengths, max_len=2048, block=BLOCK):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_paged_vs_contiguous(f):
    c, p = f([100, 300, 1500, 40], 2048)
    assert abs(c - (1 - 1940 / 8192)) < 1e-9
    assert p < 0.03, "paging wastes at most one partial block per request"

check("paged_vs_contiguous", _test_paged_vs_contiguous, paged_vs_contiguous)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: How many blocks?
def blocks_needed_solution(n, block=BLOCK):
    b = math.ceil(n / block)
    return b, b * block - n

check("blocks_needed (reference)", _test_blocks_needed, blocks_needed_solution)

In [ ]:
# @title Solution for Task 2: A block pool with ref counts and an LRU free queue
class BlockPool_solution:
    def __init__(self, num_blocks):
        self.free_q = OrderedDict((i, None) for i in range(num_blocks))
        self.ref = [0] * num_blocks
    def allocate(self, k):
        if len(self.free_q) < k: return None
        ids = []
        for _ in range(k):
            i, _ = self.free_q.popitem(last=False); self.ref[i] = 1; ids.append(i)
        return ids
    def share(self, ids):
        for i in ids:
            if self.ref[i] == 0: self.free_q.pop(i, None)
            self.ref[i] += 1
    def free(self, ids):
        for i in reversed(ids):
            self.ref[i] -= 1
            if self.ref[i] == 0: self.free_q[i] = None
    def num_free(self):
        return len(self.free_q)

check("BlockPool (reference)", _test_BlockPool, BlockPool_solution)

In [ ]:
# @title Solution for Task 3: Paged vs contiguous reservation
def paged_vs_contiguous_solution(lengths, max_len=2048, block=BLOCK):
    used = sum(lengths)
    paged = sum(math.ceil(n / block) * block for n in lengths)
    return 1 - used / (max_len * len(lengths)), 1 - used / paged

check("paged_vs_contiguous (reference)", _test_paged_vs_contiguous, paged_vs_contiguous_solution)

---
## Experiments
**Fragmentation at scale.** Draw 1,000 requests with output lengths between 50 and 2,000 tokens and compare waste. The PagedAttention paper (Kwon et al., 2023) measured that earlier systems used only 20.4%–38.2% of KV cache memory for actual token states, versus 96.3% for vLLM.

In [ ]:
import random
r = random.Random(0); L = [r.randint(50, 2000) for _ in range(1000)]
c, p = paged_vs_contiguous_solution(L, 2048)
print(f"contiguous max_len reservation wastes {c:.1%}; paged (16-token blocks) wastes {p:.2%}")
for bs in [8, 16, 32, 128]:
    print(f"block size {bs:3d}: paged waste {paged_vs_contiguous_solution(L, 2048, bs)[1]:.2%}")

---
## Takeaways
* vLLM splits the KV cache into fixed-size blocks (16 tokens by default) and gives each request a block table, like OS virtual memory pages.
* Reference counts let several requests share blocks (cached prefixes, parallel sampling) without copying.
* Freed blocks join the back of an LRU free queue, in reverse order, so cached content survives as long as possible.